# AI Innovators Lab 03: SMS Spam Detection

This version includes detailed line by line comments for high school students. The goal is not only to run the code, but also to understand how the dataset, text vectorizer, machine learning model, evaluation, and prediction steps work.


In [ ]:
# ============================================================
# AI Innovators Lab: Spam Detection
# This notebook teaches an AI model to decide whether a text
# message is "spam" or "ham."
# "Spam" means an unwanted or suspicious message.
# "Ham" means a normal message that is not spam.
# ============================================================

# Print a welcome message so students know the notebook started.
print("Welcome to the Spam Detection Project!")

# Print the main goal of this project in simple words.
print("Goal: Build an AI model that can detect spam text messages.")


In [ ]:
# ============================================================
# Step 1: Import required libraries
# A library is a collection of pre-written code that helps us
# do common tasks without writing everything from zero.
# ============================================================

# Import pandas and give it the short name pd.
# Pandas helps us work with tables, similar to spreadsheets.
import pandas as pd

# Import NumPy and give it the short name np.
# NumPy helps us work with numbers and arrays.
import numpy as np

# Import pyplot from matplotlib and give it the short name plt.
# Matplotlib helps us draw charts and graphs.
import matplotlib.pyplot as plt

# Import zipfile so Python can open compressed .zip files.
import zipfile

# Import requests so Python can download files from the internet.
import requests

# Import os so Python can work with folders and file paths.
import os

# Import train_test_split.
# This helps us divide our dataset into training data and testing data.
from sklearn.model_selection import train_test_split

# Import CountVectorizer.
# This turns text messages into numbers that a machine learning model can understand.
from sklearn.feature_extraction.text import CountVectorizer

# Import Multinomial Naive Bayes.
# This is the AI model we will train for spam detection.
from sklearn.naive_bayes import MultinomialNB

# Import accuracy_score to measure how often the model is correct.
# Import confusion_matrix to see correct and incorrect predictions by category.
# Import classification_report to see more detailed model performance results.
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report

# Print a message to confirm that all libraries were imported correctly.
print("Libraries loaded successfully!")


In [ ]:
# ============================================================
# Step 2: Download the SMS Spam Collection dataset
# A dataset is a collection of examples used to train and test AI.
# This dataset contains text messages labeled as "ham" or "spam."
# ============================================================

# Store the website address where the dataset zip file is located.
dataset_url = "https://archive.ics.uci.edu/static/public/228/sms+spam+collection.zip"

# Store the name we want to give the downloaded zip file on our computer.
zip_file = "sms_spam_collection.zip"

# Store the folder name where we will unzip, or extract, the dataset files.
extract_folder = "sms_spam_collection"

# Use requests.get() to download the dataset from the website.
# The downloaded file content is stored inside the variable named response.
response = requests.get(dataset_url)

# Open a new file in write-binary mode.
# "wb" means we are writing bytes, which is how zip files are saved.
with open(zip_file, "wb") as file:
    # Write the downloaded zip file content into the file on our computer.
    file.write(response.content)

# Print a message so students know the download step is finished.
print("Dataset downloaded successfully!")

# Check whether the folder for the extracted dataset already exists.
if not os.path.exists(extract_folder):
    # If the folder does not exist, create it.
    os.makedirs(extract_folder)

# Open the downloaded zip file in read mode.
with zipfile.ZipFile(zip_file, "r") as zip_ref:
    # Extract all files from the zip file into our dataset folder.
    zip_ref.extractall(extract_folder)

# Print a message so students know the unzip step is finished.
print("Dataset extracted successfully!")


In [ ]:
# ============================================================
# Step 3: Load the dataset into Python
# The dataset file is named SMSSpamCollection.
# Each row has two parts:
# 1. label: "ham" or "spam"
# 2. message: the actual text message
# ============================================================

# Create the full path to the dataset file.
# os.path.join() safely combines the folder name and file name.
file_path = os.path.join(extract_folder, "SMSSpamCollection")

# Read the dataset file into a pandas table called data.
data = pd.read_csv(
    # Tell pandas where the dataset file is located.
    file_path,

    # Tell pandas that columns are separated by a tab space, not a comma.
    sep="\t",

    # Tell pandas the file does not already have column names.
    header=None,

    # Give clear names to the two columns.
    names=["label", "message"]
)

# Print a message to confirm that the dataset was loaded.
print("Dataset loaded successfully!")

# Print the total number of text messages in the dataset.
print("Number of messages:", len(data))

# Show the first five rows so students can see what the data looks like.
print(data.head())


In [ ]:
# ============================================================
# Step 4: Explore the dataset
# Before training an AI model, we should look at the data.
# This helps us understand how many spam and ham messages we have.
# ============================================================

# Print a blank line, then print a title for the category counts.
print("\nNumber of messages by category:")

# Count how many times each label appears in the label column.
# This tells us how many ham and spam messages are in the dataset.
print(data["label"].value_counts())

# Create a new chart area that is 6 inches wide and 4 inches tall.
plt.figure(figsize=(6, 4))

# Count the labels and draw the counts as a bar chart.
data["label"].value_counts().plot(kind="bar")

# Add a title at the top of the chart.
plt.title("Number of Spam and Ham Messages")

# Add a label under the x-axis.
plt.xlabel("Message Type")

# Add a label beside the y-axis.
plt.ylabel("Count")

# Display the chart on the screen.
plt.show()


In [ ]:
# ============================================================
# Step 5: Convert labels into numbers
# Machine learning models work with numbers better than words.
# So we convert the text labels into numeric labels:
# ham  becomes 0
# spam becomes 1
# ============================================================

# Create a new column named label_number.
# The map() function replaces each label with a number.
data["label_number"] = data["label"].map({
    # Normal messages, called ham, are represented by the number 0.
    "ham": 0,

    # Spam messages are represented by the number 1.
    "spam": 1
})

# Show the first five rows so students can see the new numeric label column.
print(data.head())


In [ ]:
# ============================================================
# Step 6: Split the data into training and testing sets
# Training data teaches the AI model.
# Testing data checks whether the model learned well.
# We do not test the model using the same examples it trained on.
# ============================================================

# Store all text messages in X.
# X usually means the input features, or information given to the model.
X = data["message"]

# Store the numeric labels in y.
# y usually means the answer the model should learn to predict.
y = data["label_number"]

# Split the messages and labels into training and testing parts.
X_train, X_test, y_train, y_test = train_test_split(
    # These are the input messages we want to split.
    X,

    # These are the correct answers connected to those messages.
    y,

    # Use 20 percent of the dataset for testing.
    # The remaining 80 percent will be used for training.
    test_size=0.2,

    # random_state makes the split repeatable.
    # Students will get the same result each time they run the notebook.
    random_state=42,

    # stratify=y keeps the spam/ham balance similar in training and testing.
    stratify=y
)

# Print how many messages are used to train the model.
print("Training messages:", len(X_train))

# Print how many messages are saved for testing the model.
print("Testing messages:", len(X_test))


In [ ]:
# ============================================================
# Step 7: Convert text into numbers
# A computer does not understand text the way humans do.
# CountVectorizer turns messages into word-count numbers.
#
# Simple idea:
# Message: "win money now"
# Computer version: numbers showing which words appear and how often
# ============================================================

# Create a CountVectorizer object.
# This object will learn the vocabulary, or list of words, from training messages.
vectorizer = CountVectorizer(
    # lowercase=True changes all words to lowercase.
    # This helps the computer treat "Win" and "win" as the same word.
    lowercase=True,

    # stop_words="english" removes very common English words like "the" and "is."
    # These words often do not help much with spam detection.
    stop_words="english"
)

# Learn the vocabulary from the training messages and convert those messages into numbers.
# fit_transform() does two things:
# 1. fit: learn the words from X_train
# 2. transform: turn X_train into numeric word-count features
X_train_vectorized = vectorizer.fit_transform(X_train)

# Convert the test messages into numbers using the same vocabulary learned from training.
# We use transform(), not fit_transform(), because the test data should not teach new words to the model.
X_test_vectorized = vectorizer.transform(X_test)

# Print a message to confirm that text has been converted into numbers.
print("Text converted into numbers successfully!")

# Print how many unique words the vectorizer learned from the training messages.
print("Number of unique words learned:", len(vectorizer.get_feature_names_out()))


In [ ]:
# ============================================================
# Step 8: Build and train the AI model
# We use a model called Multinomial Naive Bayes.
# It works well for text classification because it learns which
# words are more common in spam messages and ham messages.
# ============================================================

# Create the machine learning model.
# At this point, the model exists but has not learned from the data yet.
model = MultinomialNB()

# Train the model using the vectorized training messages and their correct labels.
# The model studies the word-count patterns and learns which patterns suggest spam.
model.fit(X_train_vectorized, y_train)

# Print a message to confirm that the training step is complete.
print("Model training completed!")


In [ ]:
# ============================================================
# Step 9: Evaluate the model
# Evaluation means checking how well the trained model performs.
# We use the test messages because the model did not train on them.
# ============================================================

# Ask the trained model to predict labels for the test messages.
# The answers will be 0 for ham or 1 for spam.
y_pred = model.predict(X_test_vectorized)

# Compare the model's predictions with the correct test labels.
# Accuracy means the percentage of predictions the model got right.
accuracy = accuracy_score(y_test, y_pred)

# Convert the accuracy to a percentage and print it.
print("Model Accuracy:", round(accuracy * 100, 2), "%")

# Print a detailed report.
# Precision, recall, and F1-score give more information than accuracy alone.
print("\nClassification Report:")

# Show the classification report with readable class names.
print(classification_report(y_test, y_pred, target_names=["Ham", "Spam"]))

# Print a title for the confusion matrix.
print("\nConfusion Matrix:")

# Print the confusion matrix.
# It shows correct and incorrect predictions for ham and spam.
print(confusion_matrix(y_test, y_pred))


In [ ]:
# ============================================================
# Step 10: Visualize the confusion matrix
# A confusion matrix helps us see the model's mistakes.
#
# Top-left: Ham correctly predicted as Ham
# Top-right: Ham incorrectly predicted as Spam
# Bottom-left: Spam incorrectly predicted as Ham
# Bottom-right: Spam correctly predicted as Spam
# ============================================================

# Create the confusion matrix from the correct labels and predicted labels.
cm = confusion_matrix(y_test, y_pred)

# Create a new chart area that is 5 inches wide and 4 inches tall.
plt.figure(figsize=(5, 4))

# Show the confusion matrix as an image.
# Darker or lighter blocks represent different numbers.
plt.imshow(cm)

# Add a title to the chart.
plt.title("Confusion Matrix")

# Add a color bar to show what the colors mean.
plt.colorbar()

# Put labels under the x-axis positions 0 and 1.
plt.xticks([0, 1], ["Ham", "Spam"])

# Put labels beside the y-axis positions 0 and 1.
plt.yticks([0, 1], ["Ham", "Spam"])

# Label the x-axis as the model's predicted answer.
plt.xlabel("Predicted Label")

# Label the y-axis as the true correct answer.
plt.ylabel("True Label")

# Loop through each row of the confusion matrix.
for i in range(2):
    # Loop through each column of the confusion matrix.
    for j in range(2):
        # Write the number inside the matching box on the chart.
        # j controls the left-right position, and i controls the up-down position.
        plt.text(j, i, cm[i, j], ha="center", va="center")

# Display the chart.
plt.show()


In [ ]:
# ============================================================
# Step 11: Test the model with custom messages
# Students can change these messages or add their own examples.
# This helps students see how the trained model responds to new text.
# ============================================================

# Create a list of messages we want the model to classify.
custom_messages = [
    # This message sounds like a prize scam, so it may be spam.
    "Congratulations! You won $1000. Click here to claim your prize.",

    # This message sounds like a normal conversation, so it may be ham.
    "Hey, are we still meeting at 5 PM today?",

    # This message uses urgent and reward language, so it may be spam.
    "URGENT! Your account has been selected for a free reward.",

    # This message sounds like a normal school-related request, so it may be ham.
    "Can you send me the homework file?",

    # This message says the user won a phone and asks for a reply, so it may be spam.
    "You have won a brand new phone. Reply WIN now."
]

# Convert the custom messages into numbers using the same vectorizer.
# The model can only predict after the text is converted into numeric features.
custom_messages_vectorized = vectorizer.transform(custom_messages)

# Ask the model to predict whether each custom message is ham or spam.
custom_predictions = model.predict(custom_messages_vectorized)

# Ask the model for prediction probabilities.
# These probabilities help us estimate how confident the model is.
custom_probabilities = model.predict_proba(custom_messages_vectorized)

# Print a title before showing the predictions.
print("Custom Message Predictions:\n")

# Go through each message, prediction, and probability together.
# zip() lets us read matching items from multiple lists at the same time.
for message, prediction, probability in zip(custom_messages, custom_predictions, custom_probabilities):
    # If prediction is 1, label it as SPAM.
    # If prediction is 0, label it as HAM / NOT SPAM.
    label = "SPAM" if prediction == 1 else "HAM / NOT SPAM"

    # Find the highest probability and convert it into a percentage.
    confidence = max(probability) * 100

    # Print the original message.
    print("Message:", message)

    # Print the model's predicted label.
    print("Prediction:", label)

    # Print how confident the model is in its prediction.
    print("Confidence:", round(confidence, 2), "%")

    # Print a line to separate each example and make the output easier to read.
    print("-" * 60)


In [ ]:
# ============================================================
# Step 12: Let students type their own message
# This cell allows students to interact with the AI model.
# They can type any text message and see the model's prediction.
# ============================================================

# Ask the student to type a message.
# The input() function waits for the student to type something and press Enter.
student_message = input("Type a message to test: ")

# Convert the student's message into numbers.
# We put student_message inside a list because vectorizer.transform() expects a list of messages.
student_message_vectorized = vectorizer.transform([student_message])

# Ask the model to predict whether the student's message is ham or spam.
# [0] gets the first prediction because we only entered one message.
student_prediction = model.predict(student_message_vectorized)[0]

# Ask the model for the probability scores for ham and spam.
# [0] gets the first probability row because we only entered one message.
student_probability = model.predict_proba(student_message_vectorized)[0]

# Check whether the model predicted spam.
if student_prediction == 1:
    # If the prediction is 1, print that the message is spam.
    print("\nThe AI predicts: SPAM")
else:
    # If the prediction is 0, print that the message is ham, or not spam.
    print("\nThe AI predicts: HAM / NOT SPAM")

# Print the model's confidence as a percentage.
# max(student_probability) selects the larger probability value.
print("Confidence:", round(max(student_probability) * 100, 2), "%")


In [ ]:
# ============================================================
# Step 13: Show important spam-related words
# This helps students understand the model instead of treating it
# like a mystery box.
# We will find words that are more strongly connected with spam.
# ============================================================

# Get the list of words learned by the vectorizer.
# We convert it to a NumPy array so we can select words using numeric indexes.
feature_names = np.array(vectorizer.get_feature_names_out())

# Get the model's learned word scores for spam messages.
# Class 1 means spam.
spam_word_scores = model.feature_log_prob_[1]

# Get the model's learned word scores for ham messages.
# Class 0 means ham.
ham_word_scores = model.feature_log_prob_[0]

# Calculate how much more strongly each word is connected with spam than ham.
# A larger score means the word is more associated with spam.
spam_indicator_scores = spam_word_scores - ham_word_scores

# Sort the spam association scores and select the indexes of the top 20 words.
# argsort() returns indexes from smallest score to largest score.
top_spam_indices = np.argsort(spam_indicator_scores)[-20:]

# Use the top indexes to get the actual top spam-related words.
top_spam_words = feature_names[top_spam_indices]

# Use the same indexes to get the scores for those words.
top_spam_scores = spam_indicator_scores[top_spam_indices]

# Create a new chart area that is 8 inches wide and 6 inches tall.
plt.figure(figsize=(8, 6))

# Draw a horizontal bar chart of the top spam-related words.
plt.barh(top_spam_words, top_spam_scores)

# Add a title to the chart.
plt.title("Words Strongly Associated with Spam")

# Label the x-axis.
plt.xlabel("Spam Association Score")

# Label the y-axis.
plt.ylabel("Words")

# Display the chart.
plt.show()

# Print a title before listing the top spam-related words.
print("Top words associated with spam:")

# Print the words in reverse order so the strongest spam words appear first.
for word in top_spam_words[::-1]:
    # Print each word with a dash in front of it.
    print("-", word)


In [ ]:
# ============================================================
# Step 14: Save the model and vectorizer
# Saving allows us to reuse the trained model later.
# For text AI, we must save both:
# 1. the model, which makes predictions
# 2. the vectorizer, which converts text into numbers
# ============================================================

# Import pickle.
# Pickle helps Python save objects, such as trained models, into files.
import pickle

# Open a new file for the trained model in write-binary mode.
with open("spam_detection_model.pkl", "wb") as file:
    # Save the trained spam detection model into the file.
    pickle.dump(model, file)

# Open a new file for the vectorizer in write-binary mode.
with open("text_vectorizer.pkl", "wb") as file:
    # Save the vectorizer so future messages can be converted the same way.
    pickle.dump(vectorizer, file)

# Print the name of the saved model file.
print("Model saved as spam_detection_model.pkl")

# Print the name of the saved vectorizer file.
print("Vectorizer saved as text_vectorizer.pkl")


In [ ]:
# ============================================================
# Step 15: Reflection questions for final showcase
# These questions help students explain their project clearly.
# They can use these answers for a poster, presentation, or demo.
# ============================================================

# Create a list of reflection questions.
# Each question helps students think about one important part of the project.
reflection_questions = [
    # Ask students to explain the real-world problem.
    "1. What problem does your AI model solve?",

    # Ask students to identify the dataset.
    "2. What dataset did you use?",

    # Ask students to explain the two labels.
    "3. What is the difference between spam and ham?",

    # Ask students to explain text vectorization in simple words.
    "4. How did the AI convert text into numbers?",

    # Ask students to report the model's performance.
    "5. What was your model accuracy?",

    # Ask students to connect model behavior with important words.
    "6. What types of words appeared important for spam detection?",

    # Ask students to think about errors and limitations.
    "7. Did the model make any mistakes?",

    # Ask students to connect the project to real life.
    "8. How could this type of AI be used in real life?",

    # Ask students to think about responsible AI use.
    "9. What are the risks of using AI for message filtering?"
]

# Print a blank line and a title before showing the questions.
print("\nReflection Questions:")

# Loop through the list of reflection questions.
for question in reflection_questions:
    # Print each question on its own line.
    print(question)


# Presentation Template for Students

Students can copy this structure into a slide deck or poster.

**Project Title:**  
Spam Detection AI Model

**Team Members:**  
[Names]

**Problem:**  
We wanted to build an AI model that can identify whether a text message is spam or not spam.

**Dataset:**  
We used the UCI SMS Spam Collection dataset.  
Dataset Link: https://archive.ics.uci.edu/dataset/228/sms+spam+collection

**How It Works:**  
The model learns from labeled text messages. It looks at words and patterns that often appear in spam messages.

**Model Used:**  
Multinomial Naive Bayes

**Result:**  
Our model reached approximately ____% accuracy.

**Demo:**  
We typed a new message, and the model predicted whether it was spam or not spam.

**What We Learned:**  
We learned how AI can classify text messages using data.

**Future Improvement:**  
We could train with more recent spam examples, use better text features, or test messages from different languages.
